In [ ]:

import torch
import torch.nn.functional as F
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from utils.config import Config
from data_handler import DataHandler
# from classificators.dummy_classifier import DummyClassifier
# from classificators.random_forest_classifier import RandomForestClassifierSK
# from utils.utils import calculate_mcc_multilabel, plot_per_class_confusion

# pd.options.display.float_format = '{:.6f}'.format


In [ ]:
# Seeding
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
config = Config()
print(config.data)
print(config.prep)

# if you use any other libraries that require seeding, set it here as well (e.g., torch.manual_seed(SEED) for PyTorch)
# -> your results should be reproducible across runs with the same seed


val_mccs = []
test_mccs = []
lr_histories_by_fold = {}

# load data
datahandler = DataHandler(config=config)



In [ ]:
# =========================================================
# HELPERS
# =========================================================
def print_shapes(name, X, y=None):
    if y is None:
        print(f"{name}: {X.shape}, dtype={X.dtype}")
    else:
        print(f"{name}: X={X.shape}, y={y.shape}, X_dtype={X.dtype}, y_dtype={y.dtype}")
        

In [ ]:
# =========================================================
# 1. REDUCE RAW SIGNAL TO (N, 7, 16)
# =========================================================
def reduce_raw_to_16(X, out_points=16, chunk_size=10000):
    chunks = []

    for start in range(0, X.shape[0], chunk_size):
        end = min(start + chunk_size, X.shape[0])

        chunk = torch.from_numpy(X[start:end]).to(device=device, dtype=torch.float32)
        chunk_reduced = F.adaptive_avg_pool1d(chunk, out_points)   # (B, 7, 16)

        chunks.append(chunk_reduced.cpu().numpy())

        print(f"Raw chunk {start}:{end} -> {tuple(chunk_reduced.shape)}")

        del chunk, chunk_reduced
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(chunks, axis=0).astype(np.float32)


In [ ]:
# =========================================================
# 2. REDUCE FFT TO (N, 7, 16)
# =========================================================
def compute_reduced_rfft_features(X, out_bins=16, chunk_size=10000, use_log=True):
    fft_chunks = []

    for start in range(0, X.shape[0], chunk_size):
        end = min(start + chunk_size, X.shape[0])

        chunk = torch.from_numpy(X[start:end]).to(device=device, dtype=torch.float32)

        chunk_fft = torch.fft.rfft(chunk, dim=-1)   # (B, 7, 81)
        chunk_fft = torch.abs(chunk_fft).float()

        if use_log:
            chunk_fft = torch.log1p(chunk_fft)

        chunk_fft_reduced = F.adaptive_avg_pool1d(chunk_fft, out_bins)   # (B, 7, 16)

        fft_chunks.append(chunk_fft_reduced.cpu().numpy())

        print(f"FFT chunk {start}:{end} -> {tuple(chunk_fft_reduced.shape)}")

        del chunk, chunk_fft, chunk_fft_reduced
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(fft_chunks, axis=0).astype(np.float32)


In [ ]:
# =========================================================
# 3. ENTROPY OF 16 LOCAL CHUNKS -> (N, 7, 16)
# =========================================================
def compute_entropy_16_features(X, bins=16, chunk_size=10000):
    """
    Input:
        X shape = (N, 7, 160)

    Output:
        entropy_features shape = (N, 7, 16)

    Method:
        Split each 160-sample window into 16 chunks of length 10.
        Compute histogram entropy for each chunk.
    """
    assert X.shape[2] == 160, "Expected last dimension = 160"
    assert 160 % 16 == 0, "Window length must be divisible by 16"

    sub_len = 160 // 16   # 10
    entropy_chunks = []

    for start in range(0, X.shape[0], chunk_size):
        end = min(start + chunk_size, X.shape[0])

        chunk = torch.from_numpy(X[start:end]).to(device=device, dtype=torch.float32)  # (B, 7, 160)
        B, C, T = chunk.shape

        # reshape into 16 local segments of length 10
        chunk_reshaped = chunk.view(B, C, 16, sub_len)   # (B, 7, 16, 10)

        x_min = chunk_reshaped.min(dim=-1, keepdim=True).values
        x_max = chunk_reshaped.max(dim=-1, keepdim=True).values
        x_range = x_max - x_min

        valid = x_range > 0

        # normalize to [0,1]
        scaled = torch.where(valid, (chunk_reshaped - x_min) / x_range, torch.zeros_like(chunk_reshaped))

        # bin indices
        bin_idx = torch.floor(scaled * bins).long()
        bin_idx = torch.clamp(bin_idx, 0, bins - 1)   # (B, 7, 16, 10)

        # histogram counts
        counts = torch.zeros((B, C, 16, bins), device=device, dtype=torch.float32)
        counts.scatter_add_(
            dim=-1,
            index=bin_idx,
            src=torch.ones_like(bin_idx, dtype=torch.float32)
        )  # (B, 7, 16, bins)

        probs = counts / sub_len

        entropy = -torch.sum(
            torch.where(probs > 0, probs * torch.log2(probs), torch.zeros_like(probs)),
            dim=-1
        )  # (B, 7, 16)

        entropy = torch.where(valid.squeeze(-1), entropy, torch.zeros_like(entropy))

        entropy_chunks.append(entropy.cpu().numpy())

        print(f"Entropy chunk {start}:{end} -> {tuple(entropy.shape)}")

        del chunk, chunk_reshaped, x_min, x_max, x_range, valid, scaled, bin_idx, counts, probs, entropy
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(entropy_chunks, axis=0).astype(np.float32)

In [ ]:
def merge_raw_and_fft(X_raw, X_fft):
    """
    Concatenate raw data and FFT data along channel axis.

    Input:
        X_raw shape = (N, 7, 160)
        X_fft shape = (N, 7, 160)

    Output:
        X_merged shape = (N, 14, 160)
    """
    return np.concatenate([X_raw, X_fft], axis=1).astype(np.float16)



In [ ]:
# =========================================================
# 4. BUILD FINAL 21x16 FEATURES
# =========================================================
def build_21x16_features(X, chunk_size=10000, fft_log=True, entropy_bins=16):
    """
    Input:
        X shape = (N, 7, 160)

    Output:
        X_21x16 shape = (N, 21, 16)
    """
    raw_16 = reduce_raw_to_16(X, out_points=16, chunk_size=chunk_size)  # (N, 7, 16)
    fft_16 = compute_reduced_rfft_features(X, out_bins=16, chunk_size=chunk_size, use_log=fft_log)  # (N, 7, 16)
    ent_16 = compute_entropy_16_features(X, bins=entropy_bins, chunk_size=chunk_size)  # (N, 7, 16)

    X_21x16 = np.concatenate([raw_16, fft_16, ent_16], axis=1).astype(np.float32)
    return X_21x16


In [ ]:
def flatten_features(X):
    """
    Flatten 3D tensor into 2D for classical ML.

    Input:
        (N, 14, 160)s

    Output:
        (N, 14*160)
    """
    return X.reshape(X.shape[0], -1).astype(np.float16)



In [ ]:

# Leave-one-out: EXPERIMENT_ID = 1..4
for fold in range(1, 2):
    val_id = fold + 1 if fold < 4 else 1

    datahandler.config.data.test_experiment_id = fold
    # validation hat to be different from test
    datahandler.config.data.validation_experiment_id = val_id

    (train_x,train_y),(test_x,test_y),(val_x,val_y), target_vals = datahandler.get_data_loaders()
    print(train_x.shape, train_y.shape)
    print(test_x.shape, test_y.shape)
    print(val_x.shape, val_y.shape)
    # Ensure float32 to reduce memory
    train_x = train_x[:, :, :].astype(np.float16)
    # val_x   = val_x.astype(np.float16)
    # test_x  = test_x.astype(np.float16)

    print("\nOriginal shapes:")
    print_shapes("train", train_x, train_y)
    # print_shapes("val", val_x, val_y)
    # print_shapes("test", test_x, test_y)
    print("Target columns:", target_vals)

    # -----------------------------------------------------
    # FFT WITH SAME LENGTH
    # Input  shape: (N, 7, 160)
    # Output shape: (N, 7, 160)
    # -----------------------------------------------------
    print("\nComputing train FFT...")

    train_x_21x16 = build_21x16_features(train_x, chunk_size=10000, fft_log=True, entropy_bins=16)
    # val_x_21x16   = build_21x16_features(val_x, chunk_size=10000, fft_log=True, entropy_bins=16)
    # test_x_21x16  = build_21x16_features(test_x, chunk_size=10000, fft_log=True, entropy_bins=16)

    print("train_x_21x16:", train_x_21x16.shape)
    # print("val_x_21x16:", val_x_21x16.shape)
    # print("test_x_21x16:", test_x_21x16.shape)

    

    # -----------------------------------------------------
    # OPTIONAL FLATTENING FOR CLASSICAL ML
    # -----------------------------------------------------
    # train_x_flat = flatten_features(train_x_merged)
    # val_x_flat   = flatten_features(val_x_merged)
    # test_x_flat  = flatten_features(test_x_merged)

    # print("\nFlattened shapes:")
    # print_shapes("train_x_flat", train_x_flat, train_y)
    # print_shapes("val_x_flat", val_x_flat, val_y)
    # print_shapes("test_x_flat", test_x_flat, test_y)


In [1]:

# =========================================================
# 1. SPLIT FEATURES
# =========================================================
def split_features(X):
    """
    Input:
        X shape = (N, 21, 16)

    Output:
        raw, fft, entropy each (N, 7, 16)
    """
    raw = X[:, 0:7, :]
    fft = X[:, 7:14, :]
    entropy = X[:, 14:21, :]
    return raw, fft, entropy


In [ ]:

# =========================================================
# 2. HEATMAP OVER FULL N FOR ONE SENSOR
# =========================================================
def plot_sensor_over_full_N_heatmaps(X, sensor_names, sensor_idx=0, max_points=None):
    """
    Plots raw / fft / entropy over the full sample axis N for one sensor.

    X shape = (N, 21, 16)

    For readability:
    - x-axis = sample/time index over N
    - y-axis = 16 reduced bins
    """
    raw, fft, entropy = split_features(X)

    if max_points is not None and raw.shape[0] > max_points:
        idx = np.linspace(0, raw.shape[0] - 1, max_points).astype(int)
        raw = raw[idx]
        fft = fft[idx]
        entropy = entropy[idx]

    raw_s = raw[:, sensor_idx, :]       # (N, 16)
    fft_s = fft[:, sensor_idx, :]       # (N, 16)
    ent_s = entropy[:, sensor_idx, :]   # (N, 16)

    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

    sns.heatmap(raw_s.T, ax=axes[0], cmap="viridis", cbar=True)
    axes[0].set_title(f"Raw over full N - {sensor_names[sensor_idx]}")
    axes[0].set_ylabel("16 bins")

    sns.heatmap(fft_s.T, ax=axes[1], cmap="viridis", cbar=True)
    axes[1].set_title(f"FFT over full N - {sensor_names[sensor_idx]}")
    axes[1].set_ylabel("16 bins")

    sns.heatmap(ent_s.T, ax=axes[2], cmap="viridis", cbar=True)
    axes[2].set_title(f"Entropy over full N - {sensor_names[sensor_idx]}")
    axes[2].set_ylabel("16 bins")
    axes[2].set_xlabel("Sample index over N")

    plt.tight_layout()
    plt.show()




In [ ]:
# =========================================================
# 3. LINE PLOT OF BIN-WISE MEAN OVER N FOR ONE SENSOR
# =========================================================
def plot_sensor_mean_over_full_N(X, sensor_names, sensor_idx=0):
    """
    For one sensor, average the 16 bins at each sample.
    Result is one scalar per sample for raw / fft / entropy.
    """
    raw, fft, entropy = split_features(X)

    raw_mean = raw[:, sensor_idx, :].mean(axis=1)      # (N,)
    fft_mean = fft[:, sensor_idx, :].mean(axis=1)      # (N,)
    ent_mean = entropy[:, sensor_idx, :].mean(axis=1)  # (N,)

    plt.figure(figsize=(16, 5))
    plt.plot(raw_mean, label="Raw mean over 16 bins")
    plt.plot(fft_mean, label="FFT mean over 16 bins")
    plt.plot(ent_mean, label="Entropy mean over 16 bins")
    plt.title(f"Mean feature trend over full N - {sensor_names[sensor_idx]}")
    plt.xlabel("Sample index over N")
    plt.ylabel("Value")
    plt.legend()
    plt.grid()
    plt.show()


    plt.show()


In [ ]:

# =========================================================
# 4. PLOT ALL 16 BINS AS LINES OVER FULL N FOR ONE SENSOR
# =========================================================
def plot_sensor_bins_over_full_N(X, sensor_names, sensor_idx=0, feature_type="raw", max_points=5000):
    """
    Plot all 16 bins as lines over N for one sensor.
    Since N may be huge, downsample for readability.
    """
    raw, fft, entropy = split_features(X)

    feature_map = {
        "raw": raw,
        "fft": fft,
        "entropy": entropy
    }

    data = feature_map[feature_type][:, sensor_idx, :]  # (N, 16)

    if data.shape[0] > max_points:
        idx = np.linspace(0, data.shape[0] - 1, max_points).astype(int)
        data = data[idx]

    plt.figure(figsize=(16, 6))
    for b in range(data.shape[1]):
        plt.plot(data[:, b], label=f"bin {b}")

    plt.title(f"{feature_type.upper()} bins over full N - {sensor_names[sensor_idx]}")
    plt.xlabel("Sample index over N (downsampled if needed)")
    plt.ylabel("Value")
    plt.legend(ncol=4, fontsize=8)
    plt.grid()
    plt.show()




In [ ]:
# =========================================================
# 5. PLOT ALL SENSORS AS MEAN TRENDS OVER FULL N
# =========================================================
def plot_all_sensors_mean_over_full_N(X, sensor_names, feature_type="raw"):
    """
    For each sensor, plot the mean across 16 bins over full N.
    """
    raw, fft, entropy = split_features(X)

    feature_map = {
        "raw": raw,
        "fft": fft,
        "entropy": entropy
    }

    data = feature_map[feature_type]  # (N, 7, 16)
    n_sensors = data.shape[1]

    fig, axes = plt.subplots(n_sensors, 1, figsize=(16, 2.5 * n_sensors), sharex=True)

    for i in range(n_sensors):
        mean_trend = data[:, i, :].mean(axis=1)  # (N,)
        axes[i].plot(mean_trend)
        axes[i].set_title(f"{feature_type.upper()} mean over N - {sensor_names[i]}")
        axes[i].set_ylabel("Value")
        axes[i].grid()

    axes[-1].set_xlabel("Sample index over N")
    plt.tight_layout()
    plt.show()




In [ ]:
# =========================================================
# 6. CLASS-WISE FULL-N HEATMAP
# =========================================================
def plot_classwise_sensor_heatmap(X, y, sensor_names, sensor_idx=0, feature_type="raw"):
    """
    Sort samples by class and plot one sensor as a heatmap over N.
    """
    raw, fft, entropy = split_features(X)

    feature_map = {
        "raw": raw,
        "fft": fft,
        "entropy": entropy
    }

    labels = np.argmax(y, axis=1)
    order = np.argsort(labels)

    data = feature_map[feature_type][order, sensor_idx, :]  # (N, 16)

    plt.figure(figsize=(16, 5))
    sns.heatmap(data.T, cmap="viridis", cbar=True)
    plt.title(f"{feature_type.upper()} over sorted class order - {sensor_names[sensor_idx]}")
    plt.xlabel("Samples sorted by class")
    plt.ylabel("16 bins")
    plt.show()



In [ ]:

# =========================================================
# USAGE
# =========================================================
X = train_x_21x16   # shape (N, 21, 16)
y = train_y         # shape (N, num_classes)
sensor_names = datahandler.config.data.sensor_cols

# Example:
raw, fft, entropy = split_features(train_x_21x16)


In [ ]:
# 1) Best overview for one sensor across full N
plot_sensor_over_full_N_heatmaps(train_x_21x16, sensor_names, sensor_idx=0, max_points=5000)


In [ ]:
# 2) One line per feature group across full N
plot_sensor_mean_over_full_N(train_x_21x16, sensor_names, sensor_idx=0)



In [ ]:
# 3) All 16 bins as lines across N
plot_sensor_bins_over_full_N(train_x_21x16, sensor_names, sensor_idx=0, feature_type="raw", max_points=5000)
plot_sensor_bins_over_full_N(train_x_21x16, sensor_names, sensor_idx=0, feature_type="fft", max_points=5000)
plot_sensor_bins_over_full_N(train_x_21x16, sensor_names, sensor_idx=0, feature_type="entropy", max_points=5000)



In [ ]:
# 4) All sensors mean trend across N
plot_all_sensors_mean_over_full_N(train_x_21x16, sensor_names, feature_type="raw")
plot_all_sensors_mean_over_full_N(train_x_21x16, sensor_names, feature_type="fft")
plot_all_sensors_mean_over_full_N(train_x_21x16, sensor_names, feature_type="entropy")



In [ ]:
# 5) Classwise sorted heatmap
plot_classwise_sensor_heatmap(train_x_21x16, train_y, sensor_names, sensor_idx=0, feature_type="raw")